# GrainTrace - Experiment

# 1. Purpose

This notebook is dedicated to perform metric/embeddings learning on stringed instruments' bow images to create unique digital fingerprint for each.

Finetuned Model: ResNet50

Loss: Triplet Hard Batch Mining 

Distance metric: Euclidean Distance

# 2. Imports

In [2]:
# ========================================
# Standard Library
# ========================================
import os
import random
import zipfile
from collections import defaultdict
from pathlib import Path

# ========================================
# Data Manipulation & Numerical Computing
# ========================================
import numpy as np
import pandas as pd

# ========================================
# Visualization
# ========================================
import matplotlib.pyplot as plt
import plotly.express as px

# ========================================
# Image Processing
# ========================================
from PIL import Image

# ========================================
# Progress Tracking
# ========================================
from tqdm.auto import tqdm

# ========================================
# PyTorch Core
# ========================================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ========================================
# PyTorch Data Utilities
# ========================================
from torch.utils.data import (
    Dataset,
    DataLoader,
    Sampler,
    BatchSampler,
)

# ========================================
# TorchVision
# ========================================
from torchvision import transforms, models

# ========================================
# Scikit-learn
# ========================================
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    pairwise_distances,
)
from sklearn.model_selection import train_test_split

# ========================================
# Dimensionality Reduction
# ========================================
import umap.umap_ as umap

2026-07-05 23:09:33.677376: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783292973.891178      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783292973.955003      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783292974.466988      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783292974.467031      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1783292974.467034      58 computation_placer.cc:177] computation placer alr

# 3. Preprocessing & Data Augmentation

In [3]:
transform_resize = transforms.Resize((224 * 3, 224 * 3))
transform_normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
transforms.RandomResizedCrop(size=384, scale=(0.5, 1.0),ratio=(0.9, 1.1))

train_transform_2 = transforms.Compose([
    transforms.RandomCrop(448),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

transform_1 = transforms.Compose([
    transform_resize,
    transforms.ToTensor(),
    transform_normalize
])

val_transform = transforms.Compose([
    transform_resize,
    transforms.ToTensor(),
    transform_normalize
])

In [4]:
CONFIG={"BATCH_SIZE": 18,
        "DEVICE": "cuda" if torch.cuda.is_available() else "cpu"}

# 4. Dataset

### 4.1 Bow Number-Image Paths Pairs Creation

In [5]:
def datasetcreationhelper(dataset_path):
    grouped_images = defaultdict(list)
    for bow_folder in os.listdir(dataset_path):
        bow_path = os.path.join(dataset_path, bow_folder) # Each Bow Folder
        if not os.path.isdir(bow_path):
            continue

        for image_name in os.listdir(bow_path):
            # STRICT FILTER: Only load images with 'RSU' in the name
           

            image_path = os.path.join(bow_path, image_name)
            grouped_images[bow_folder].append(image_path)

    grouped_images = {k: v for k, v in grouped_images.items() if len(v) > 0}
    return grouped_images # returns paths of grouped images

In [6]:
kaggle_path = '/kaggle/input/datasets/moedataguru/mycroppeddatasetobjdet/cropped_bows_new_objdet/cropped_bows'
path = "my_finetuning/kaggle/working/MyBows_Final"
groupedimages = datasetcreationhelper(kaggle_path)

### 4.2 Triplet Hard Batch Mining Dataset

In [7]:
class HardTripletDataset(Dataset):

    def __init__(self, grouped_images, transform=None):
            self.transform = transform
            self.samples = []
            self.labels = []
    
            # create mapping: "bow_0002" -> 0, "bow_0003" -> 1, ...
            self.label_to_id = {
                label: idx for idx, label in enumerate(sorted(grouped_images.keys()))
            }
    
            for label, paths in grouped_images.items():
                label_id = self.label_to_id[label]
    
                for path in paths:
                    self.samples.append((path, label_id))
                    self.labels.append(label_id)


    def __len__(self):
        return len(self.samples) * 20

    def __getitem__(self, idx):

        path, label = self.samples[idx]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

### 4.3 Custom Batch Sampler

Ensures batch has more p classes and each class has k samples

In [8]:
class PKBatchSampler(BatchSampler):

    def __init__(self, labels, p_classes, k_samples):

        self.labels = labels
        self.p_classes = p_classes
        self.k_samples = k_samples

        self.label_to_indices = defaultdict(list)

        for idx, label in enumerate(labels):
            self.label_to_indices[label].append(idx)

        self.unique_labels = list(self.label_to_indices.keys())

    def __iter__(self):

        num_batches = len(self)

        for _ in range(num_batches):

            selected_classes = random.sample(self.unique_labels, self.p_classes)

            batch = []

            for cls in selected_classes:

                indices = self.label_to_indices[cls]

                batch.extend(random.sample(indices, self.k_samples))

            yield batch

    def __len__(self):

        batch_size = self.p_classes * self.k_samples

        return len(self.labels) // batch_size

# 5. Resnet50 Model

In [9]:
class ConResNet50(nn.Module):
    def __init__(self, projection_dim=256, freeze_backbone=True):
        super().__init__()

        self.encoder = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

        # Remove classification head
        feature_dim = self.encoder.fc.in_features  # 2048
        self.encoder.fc = nn.Identity()

        if freeze_backbone:
            # freeze everything first
            for param in self.encoder.parameters():
                param.requires_grad = False

            # unfreeze layer3
            for param in self.encoder.layer3.parameters():
                param.requires_grad = True

            # unfreeze layer4
            for param in self.encoder.layer4.parameters():
                param.requires_grad = True

        
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, 1024),
            nn.LayerNorm(1024),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(1024, projection_dim, bias=False))

    def forward(self, x):
        # Backbone feature extraction
        x = self.encoder(x)          # [B, 2048]

        # Projection head
        x = self.projector(x)        # [B, projection_dim]

        # Normalize for contrastive learning
        x = F.normalize(x, dim=1)

        return x

# 6. Triplet Hard Batch Mining Loss Function

In [10]:
class BatchHardTripletLoss(nn.Module):

    def __init__(self, margin=0.5):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):

        dist_matrix = torch.cdist(embeddings, embeddings, p=2)

        mask_pos = labels.unsqueeze(0) == labels.unsqueeze(1)
        mask_neg = ~mask_pos

        hard_pos = (dist_matrix * mask_pos.float()).max(dim=1)[0]

        dist_matrix_neg = dist_matrix.clone()
        dist_matrix_neg[mask_pos] = float('inf')
        hard_neg = dist_matrix_neg.min(dim=1)[0]

        loss = F.relu(hard_pos - hard_neg + self.margin).mean()
        return loss

# 7. Training

In [11]:
model = ConResNet50().to(CONFIG["DEVICE"])


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 208MB/s]


### 7.1 Training Configs

In [13]:
optimizer = torch.optim.AdamW(
    [{"params": model.encoder.layer3.parameters(), "lr": 1e-5,},
     {"params": model.encoder.layer4.parameters(), "lr": 1e-5,},
     {"params": model.projector.parameters(),"lr": 1e-4,}],
        weight_decay=1e-4,)

In [14]:
train_dataset_trip = HardTripletDataset(groupedimages, transform_1)
sampler = PKBatchSampler(labels=train_dataset_trip.labels, p_classes=5, k_samples=12)
#train_loader = DataLoader(train_dataset_trip, batch_size=CONFIG['BATCH_SIZE'])
train_loader = DataLoader(train_dataset_trip, batch_sampler = sampler)
criterion = BatchHardTripletLoss()

### 7.2 Training Loop

In [16]:
EPOCHS = 10 # final epoch count was 110

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")

    for images, labels in pbar:

        images = images.to(CONFIG["DEVICE"])
        labels = labels.to(CONFIG["DEVICE"])

        optimizer.zero_grad()

        proj = model(images)

        loss = criterion(proj, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}", avg_loss=f"{running_loss/(pbar.n+1):.4f}")

    checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()}

    torch.save(checkpoint, "/kaggle/working/checkpoint.pth")

    print(f"Epoch {epoch+1} | Loss: {running_loss / len(train_loader):.4f}")

Epoch 1/10:   0%|          | 0/35 [00:00<?, ?it/s]

Epoch 1 | Loss: 0.0242


Epoch 2/10:   0%|          | 0/35 [00:00<?, ?it/s]

Epoch 2 | Loss: 0.0091


Epoch 3/10:   0%|          | 0/35 [00:00<?, ?it/s]

...

Epoch 8 | Loss: 0.0073


Epoch 9/10:   0%|          | 0/35 [00:00<?, ?it/s]

Epoch 9 | Loss: 0.0005


Epoch 10/10:   0%|          | 0/35 [00:00<?, ?it/s]

Epoch 10 | Loss: 0.0011


### 7.3 Load a checkpoint or model for further training

In [12]:
# checkpoint = {
#     "epoch": epoch,
#     "model_state_dict": model.state_dict(),
#     "optimizer_state_dict": optimizer.state_dict(),
#     "loss": loss.item()
# }

# torch.save(checkpoint, "/kaggle/working/checkpoint.pth")

checkpoint = torch.load("/kaggle/input/models/moedataguru/resnetfinal/pytorch/default/1/ResnetTestIncluded.pth")

model.load_state_dict(checkpoint)
# optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

# start_epoch = checkpoint["epoch"] + 1

<All keys matched successfully>

1 Epoch: 0.2640

2 Epoch: 

checkpoint = {
    "epoch": epoch,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "loss": loss.item()
}

torch.save(checkpoint, "/kaggle/working/checkpoint.pth")

checkpoint = torch.load("/kaggle/working/checkpoint.pth")

model.load_state_dict(checkpoint["model_state_dict"])
optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

start_epoch = checkpoint["epoch"] + 1

From Ipy

# 8. Evaluation

### 8.1 Inference Dataset

In [13]:
class BowInferenceDataset(Dataset):
    def __init__(self, grouped_images, transform=None):
        self.samples = []
        self.transform = transform

        for label, paths in grouped_images.items():
            for path in paths:
                self.samples.append((path, label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]

        image = Image.open(path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label, path

In [14]:
def extract_embeddings_batched(
    grouped_images,
    model,
    transform,
    device,
    batch_size=8,
    num_workers=2
):
    dataset = BowInferenceDataset(
        grouped_images=grouped_images,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True
    )

    model.eval()

    all_embeddings = []
    all_labels = []
    all_paths = []

    with torch.no_grad():
        for images, labels, paths in tqdm(loader, desc="Extracting embeddings"):

            images = images.to(device)

            # Forward pass in batch
            embeddings = model(images)

            # Move to CPU numpy
            embeddings = embeddings.detach().cpu().numpy()

            all_embeddings.append(embeddings)
            all_labels.extend(labels)
            all_paths.extend(paths)

    all_embeddings = np.vstack(all_embeddings)

    return all_embeddings, all_labels, all_paths

### 8.2 Run Inference 

In [15]:
embeddings, labels, paths = extract_embeddings_batched(
    grouped_images=groupedimages,
    model=model,
    transform=val_transform,
    device=CONFIG["DEVICE"],
    batch_size=CONFIG["BATCH_SIZE"],  # or smaller if memory issue
    num_workers=2
)

print("Embeddings shape:", embeddings.shape)
print("Number of labels:", len(labels))
print("Number of paths:", len(paths))

Extracting embeddings:   0%|          | 0/117 [00:00<?, ?it/s]

Embeddings shape: (2105, 256)
Number of labels: 2105
Number of paths: 2105


### 8.3 Calculate Average, Minimum and Maximum Positive/Negative Embeddings' Distances 

In [16]:
def avg_positive_negative_distances(embeddings, labels, normalize=False):
    """
    Computes average Euclidean distance for:
    - positive pairs: same label
    - negative pairs: different labels

    Args:
        embeddings: numpy array or torch tensor of shape (N, D)
        labels: list/array/tensor of shape (N,)
        normalize: if True, L2-normalize embeddings before distance calculation

    Returns:
        dict with avg positive/negative distances and counts
    """

    # Convert to numpy if torch tensor
    if hasattr(embeddings, "detach"):
        embeddings = embeddings.detach().cpu().numpy()

    if hasattr(labels, "detach"):
        labels = labels.detach().cpu().numpy()

    embeddings = np.asarray(embeddings)
    labels = np.asarray(labels)

    if normalize:
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        embeddings = embeddings / np.clip(norms, 1e-8, None)

    # Pairwise Euclidean distances
    dist = pairwise_distances(embeddings, metric="euclidean")

    # Create masks
    same_label = labels[:, None] == labels[None, :]
    different_label = labels[:, None] != labels[None, :]

    # Remove diagonal/self-pairs
    not_self = ~np.eye(len(labels), dtype=bool)

    positive_mask = same_label & not_self
    negative_mask = different_label

    positive_distances = dist[positive_mask]
    negative_distances = dist[negative_mask]

    results = {
        "avg_positive_distance": positive_distances.mean(),
        "avg_negative_distance": negative_distances.mean(),
        "num_positive_pairs": len(positive_distances),
        "num_negative_pairs": len(negative_distances),
        "min_positive_distance": positive_distances.min(),
        "max_positive_distance": positive_distances.max(),
        "min_negative_distance": negative_distances.min(),
        "max_negative_distance": negative_distances.max(),
    }

    return results

In [17]:
print(avg_positive_negative_distances(embeddings, labels))

{'avg_positive_distance': np.float32(0.10705169), 'avg_negative_distance': np.float32(1.3708959), 'num_positive_pairs': 66154, 'num_negative_pairs': 4362766, 'min_positive_distance': np.float32(0.009405894), 'max_positive_distance': np.float32(0.40387267), 'min_negative_distance': np.float32(0.43992114), 'max_negative_distance': np.float32(1.9896055)}


### 8.4 Calculate Positive, Negative and Overall Accuracy

Uncomment the last block for a Visualisation of classification correctness

In [18]:
names = [path.split('/')[-1] for path in paths]
names = ['_'.join(name.split('_')[:3]) for name in names]
import numpy as np
import plotly.express as px
from sklearn.metrics import pairwise_distances

# -----------------------------
# 1. GROUP BY LABEL (correct way)
# -----------------------------
labels = np.array(labels)
names = np.array(names)

grouped_idx = []

for lbl in sorted(np.unique(labels)):
    idx_lbl = np.where(labels == lbl)[0]
    grouped_idx.extend(idx_lbl)

grouped_idx = np.array(grouped_idx)

emb_sorted = embeddings[grouped_idx]
labels_sorted = labels[grouped_idx]
names_sorted = names[grouped_idx]

# -----------------------------
# 2. DISTANCE MATRIX
# -----------------------------
dist = pairwise_distances(emb_sorted, metric="euclidean")

# -----------------------------
# 3. THRESHOLD BINARY MATRIX
# -----------------------------
threshold = 0.3  # adjust later based on distribution
binary = (dist > threshold).astype(int)


# Ground truth:
# 0 -> same bow
# 1 -> different bow
gt = (labels_sorted[:, None] != labels_sorted[None, :]).astype(int)

mask = np.triu(np.ones_like(gt, dtype=bool), k=1)

y_true = gt[mask]
y_pred = binary[mask]

# Overall accuracy
overall_acc = (y_true == y_pred).mean()

# Same bow accuracy (sensitivity)
same_mask = (y_true == 0)
same_acc = (y_pred[same_mask] == 0).mean()

# Different bow accuracy (specificity)
diff_mask = (y_true == 1)
diff_acc = (y_pred[diff_mask] == 1).mean()

# -----------------------------
# CONFUSION MATRIX (with totals)
# -----------------------------
tp = np.sum((y_true == 0) & (y_pred == 0))
fn = np.sum((y_true == 0) & (y_pred == 1))
tn = np.sum((y_true == 1) & (y_pred == 1))
fp = np.sum((y_true == 1) & (y_pred == 0))

total_same = np.sum(y_true == 0)
total_diff = np.sum(y_true == 1)

print("\nConfusion Matrix:")
print(f"TP (same correct)   : {tp} / {total_same}")
print(f"FN (same missed)    : {fn} / {total_same}")
print(f"TN (diff correct)   : {tn} / {total_diff}")
print(f"FP (diff wrong)     : {fp} / {total_diff}")

print(f"\nOverall accuracy        : {overall_acc:.4f}")
print(f"Same bow accuracy       : {same_acc:.4f}")
print(f"Different bow accuracy  : {diff_acc:.4f}")


# # -----------------------------
# # 4. AXIS LABELS (GROUPED VIEW)
# # -----------------------------
# axis_labels = [f"Bow {lbl}" for lbl in labels_sorted]

# -----------------------------
# # 5. HOVER MATRIX
# # -----------------------------
# n = len(names_sorted)
# hover = np.empty((n, n), dtype=object)

# for i in range(n):
#     for j in range(n):
#         hover[i, j] = (
#             f"{names_sorted[i]}<br>"
#             f"Bow {labels_sorted[i]}<br><br>"
#             f"{names_sorted[j]}<br>"
#             f"Bow {labels_sorted[j]}<br><br>"
#             f"Distance: {dist[i, j]:.4f}<br>"
#             f"Binary: {binary[i, j]}"
#         )

# # -----------------------------
# # 6. PLOT
# # -----------------------------
# fig = px.imshow(
#     binary,
#     color_continuous_scale=["#2ca02c", "#d62728"],  # green / red
#     title=f"Grouped Binary Embedding Matrix (threshold={threshold})"
# )

# fig.update_traces(
#     customdata=hover,
#     hovertemplate="%{customdata}<extra></extra>"
# )

# fig.update_layout(
#     width=1000,
#     height=1000
# )

# fig.update_xaxes(tickangle=90)

# fig.show()


Confusion Matrix:
TP (same correct)   : 33017 / 33077
FN (same missed)    : 60 / 33077
TN (diff correct)   : 2181383 / 2181383
FP (diff wrong)     : 0 / 2181383

Overall accuracy        : 1.0000
Same bow accuracy       : 0.9982
Different bow accuracy  : 1.0000


# 9. Save and Dowload the model

In [18]:

# Save the model weights
torch.save(model.state_dict(), 'model_weights.pth')

# Or save the entire model
torch.save(model, 'entire_model.pth')

In [19]:
from IPython.display import FileLink

# This will create a clickable link right in your notebook output
FileLink('model_weights.pth')

/kaggle/working/model_weights.pth